## ***Gold Transformation Script***

###***Storage Container Path***

In [0]:
brz_path="abfss://bronze@stgactadventureworks.dfs.core.windows.net/"
slv_path="abfss://silver@stgactadventureworks.dfs.core.windows.net/"
gld_path="abfss://gold@stgactadventureworks.dfs.core.windows.net/"

###***Schema Path***

In [0]:
Gold_schema_path=f"Adventureworks.Gold_schema"

###***Calendar***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

slv_data_path = f"{slv_path}AdventureWorks_Calendar"

df_gld_calendar = spark.read.format("delta").option("header", True).option("inferSchema", True).load(slv_data_path)
df_gld_calendar = df_gld_calendar.withColumn("_source_file", col("_metadata.file_path")).withColumn("injested_at",current_timestamp())

df_gld_calendar.write.mode('overwrite').format("delta").option("overwriteSchema", "true").option("path",f"{gld_path}gld_dim_Calendar").saveAsTable(f"{Gold_schema_path}.gld_dim_Calendar")


df_gld_calendar.limit(10).display()

###***Customers***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

slv_data_path = f"{slv_path}AdventureWorks_Customers"

df_gld_customers = spark.read.format("delta").option("header", True).option("inferSchema", True).load(slv_data_path)
df_gld_customers = df_gld_customers.withColumn("_source_file", col("_metadata.file_path")).withColumn("injested_at",current_timestamp())

df_gld_customers.write.mode('overwrite').format("delta").option("overwriteSchema", "true").option("path",f"{gld_path}gld_dim_Customers").saveAsTable(f"{Gold_schema_path}.gld_dim_Customers")


df_gld_customers.limit(10).display()

###***Products***

In [0]:
from pyspark.sql.functions import *

slv_data_path_products = f"{slv_path}AdventureWorks_Products"
slv_data_path_subcategories = f"{slv_path}AdventureWorks_Product_Subcategories"
slv_data_path_categories = f"{slv_path}AdventureWorks_Product_categories"


df_products = spark.read.format("delta").load(slv_data_path_products)
df_subcategories = spark.read.format("delta").load(slv_data_path_subcategories)
df_categories = spark.read.format("delta").load(slv_data_path_categories)

df_gld_products= (
    df_products.alias("p")
    .join(
        df_subcategories.alias("sc"),
        col("p.ProductSubcategoryKey") == col("sc.ProductSubcategoryKey"),
        "left"
    )
    .join(
        df_categories.alias("cat"),
        col("sc.ProductCategoryKey") == col("cat.ProductCategoryKey"),
        "left"
    )
    .select(
        col("p.ProductKey").alias("ProductKey"),
        col("p.ProductSKU").alias("ProductSKU"),
        col("p.ProductName").alias("ProductName"),
        col("p.ModelName").alias("ModelName"),
        col("p.ProductDescription").alias("ProductDescription"),
        col("p.ProductColor").alias("ProductColor"),
        col("p.ProductSize").alias("ProductSize"),
        col("p.ProductStyle").alias("ProductStyle"),
        col("p.ProductCost").alias("ProductCost"),
        col("p.ProductPrice").alias("ProductPrice"),
        col("sc.ProductSubcategoryKey").alias("ProductSubcategoryKey"),
        coalesce(col("sc.SubcategoryName"), lit("Unknown")).alias("SubcategoryName"),
        col("cat.ProductCategoryKey").alias("ProductCategoryKey"),
        coalesce(col("cat.CategoryName"), lit("Unknown")).alias("CategoryName"),
        current_timestamp().alias("_ingested_at")
    )
)

df_gld_products.write.mode('overwrite').format("delta").option("overwriteSchema", "true").option("path",f"{gld_path}gld_dim_Products").saveAsTable(f"{Gold_schema_path}.gld_dim_Products")

df_gld_products.display()

###***Territories***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

slv_data_path = f"{slv_path}AdventureWorks_Territories"

df_gld_territories = spark.read.format("delta").option("header", True).option("inferSchema", True).load(slv_data_path)
df_gld_territories = df_gld_territories.withColumn("_source_file", col("_metadata.file_path")).withColumn("injested_at",current_timestamp())

df_gld_territories.write.mode('overwrite').format("delta").option("overwriteSchema", "true").option("path",f"{gld_path}gld_dim_Territories").saveAsTable(f"{Gold_schema_path}.gld_dim_Territories")


df_gld_territories.limit(10).display()

###***Returns***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

slv_data_path = f"{slv_path}AdventureWorks_Returns"
df_gld_returns= spark.read.format("delta").option("header", True).option("inferSchema", True).load(slv_data_path)

df_gld_returns=df_gld_returns.withColumn("Year", year(col("ReturnDate"))).withColumn("Quarter", concat(lit("Q"), quarter(col("ReturnDate")))).withColumn('Month_name',date_format("ReturnDate", "MMMM")).withColumn('Month',month(col("ReturnDate")))

df_gld_returns.write.mode('overwrite').format("delta").option("overwriteSchema", "true").option("path",f"{gld_path}gld_fact_Returns").saveAsTable(f"{Gold_schema_path}.gld_fact_Returns")

df_gld_returns.limit(10).display()

###***Sales***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

slv_data_path = f"{slv_path}AdventureWorks_Sales"
df_gld_sales= spark.read.format("delta").option("header", True).option("inferSchema", True).load(slv_data_path)

df_gld_sales=df_gld_sales.withColumn("Year", year(col("OrderDate"))).withColumn("Quarter", concat(lit("Q"), quarter(col("OrderDate")))).withColumn('Month_name',date_format("OrderDate", "MMMM")).withColumn('Month',month(col("OrderDate")))

df_gld_sales.write.mode('overwrite').format("delta").option("overwriteSchema", "true").option("path",f"{gld_path}gld_fact_Sales").saveAsTable(f"{Gold_schema_path}.gld_fact_Sales")

df_gld_sales.limit(10).display()